<div style="background: linear-gradient(135deg, #1a1a2e 0%, #2d0a16 100%); padding: 32px; border-radius: 12px; text-align: center; font-family: monospace;">
<h1 style="color: #FFD700; font-size: 2.6em; margin: 0; letter-spacing: 3px;">🌿 THE NEIGHBOR GAMES 🌿</h1>
<h2 style="color: #ffffff; font-size: 1.2em; margin: 10px 0 0 0; font-weight: normal; letter-spacing: 1px;">k-Nearest Neighbor Prediction Championship</h2>
<p style="color: #aaaacc; margin: 10px 0 0 0; font-size: 0.9em;">Elements of Data Science · Temple University</p>
<p style="color: #FFD700; margin: 8px 0 0 0; font-size: 0.85em; font-style: italic;">"May the best features be ever in your favor."</p>
</div>

---

### 🏆 Competition Overview

| Round | Task | Challenges |
|---|---|---|
| 🏋️ Training Camp | Split and standardize the data, then build k-nearest neighbors **by hand** for one mystery molecule | 1–7 |
| 🌿 Tribute Round | Shared baseline: run as-is | — |
| ⚔️ Arena Round | Tune `k` and the features | 8 |
| 🎯 Prize Round | Predict the bp of two real molecules | 9–10 |
| 🌟 Wildcard | Inverse-distance weighting | 11 |

**How the challenges work.** Each 🖊️ challenge is **one line of code** with `...` where your team fills in the missing piece. Run it, then run the ✅ check cell under it: it tells you whether you got it, and gives a hint if not. Work it out together; everyone on the team fills in their own notebook.

After each round, click **📡 Submit** to send your team's score to the live scoreboard. Submissions open when your instructor turns the light **🟢 green**.

---
### ⚙️ Setup — run this cell first
It loads the usual tools (NumPy for arrays, `datascience` for Tables, matplotlib for plots), then connects to the class scoreboard: it finds the course's `class_games.py` in `Fall 2026/shared-public` and loads `neighbor_team()` to register your team, `neighbor_check(...)` to check a challenge, and `submit_button(...)` to send a score.

**Restarted your kernel? Run this cell again first.**

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

import sys, os
for path in ['../shared-public', '../../shared-public', '../../../shared-public', '/home/jovyan/data/Fall 2026/shared-public']:
    if os.path.exists(os.path.join(path, 'class_games.py')):
        sys.path.insert(0, os.path.abspath(path))
from class_games import neighbor_team, neighbor_check, submit_button
print("✅ Ready")

---
### 📛 Your team
Type your team's name between the quotes, and list **everyone on your team** (first name and last initial) in `team_members`. A list is names in quotes, separated by commas, inside square brackets. Everyone on the team uses the **same team name**. Add or delete `""` entries to match your team size.

In [ ]:
team_name = ""   # e.g. "District 4: The Alkanol Alliance"

In [ ]:
team_members = ["", "", ""]   # e.g. ["Ada L.", "Grace H.", "Alan T."]

Register your team with the class scorecard. It works before the competition starts. (A `NameError` here means the ⚙️ Setup cell hasn't been run yet.)

In [ ]:
neighbor_team()

---
### 🔁 Quick review: list comprehensions
Almost every challenge today uses a **list comprehension**: a one-line `for` loop that builds a list. These two cells make the same list:

In [ ]:
squares = []
for x in make_array(1, 2, 3, 4):
    squares.append(x ** 2)
squares

In [ ]:
[x ** 2 for x in make_array(1, 2, 3, 4)]

Read it as: **"`x ** 2`, for each `x` in the array."**

`[ what to compute   for name in things ]`

- **`for name in things`** goes through the items one at a time and calls each one `name`.
- **`what to compute`** is done to each item, and the results are collected in a list, in order.

Two more you'll meet. Try changing them:

In [ ]:
[mw / 2 for mw in make_array(32.04, 46.07, 60.10)]   # half of each molecular weight

In [ ]:
[i * 10 for i in np.arange(5)]                          # i is a row number: 0, 1, 2, 3, 4

More: [Inferential Thinking 9.2, Iteration](https://inferentialthinking.com/chapters/09/2/Iteration.html) (the `for` loop) and [Python's list comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions).

---
## 🏋️ Training Camp

### 📂 The data
72 alcohols with their boiling points (`bp`, in K) and three features: molecular weight (`MW`), `degree` (1°, 2° or 3° alcohol), and number of `carbons`.

In [ ]:
ROH_data = Table().read_table('data/ROH_data.csv').select('bp', 'MW', 'degree', 'carbons')
ROH_data.show(5)

### ✂️ Training and test sets
We split the molecules at random into a **training** set, the molecules we "know", and a **test** set: we pretend not to know their boiling points, predict them from the training set, then check how close we came.

`table.split(k)` shuffles the rows and returns **two** tables: `k` random rows, and all the rest. We'll train on three quarters of the 72 molecules.

`np.random.seed(42)` makes the shuffle the same in everyone's notebook, so all teams compete on the same test set.

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 1 of 11 — Split the data</strong><br>
How many molecules is three quarters of 72? <code>split</code> needs a whole number.
</div>

In [ ]:
np.random.seed(42)

In [ ]:
train_roh, test_roh = ROH_data.split(...)

In [ ]:
print(train_roh.num_rows, 'training and', test_roh.num_rows, 'test molecules')

In [ ]:
neighbor_check('split')

---
### 📏 Standard units
Molecular weight runs from about 30 to 200 g/mol, but degree is just 1, 2 or 3. Measured as they are, MW would swamp every distance. So we put every feature in **standard units**: how many SDs above or below the mean ([Inferential Thinking 14.2](https://inferentialthinking.com/chapters/14/2/Variability.html)):

$$z = \frac{x - \mu}{\sigma}$$

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 2 of 11 — The z-score formula</strong><br>
Finish the one-line function: subtract the mean <code>mu</code>, then divide by the SD <code>sigma</code>.
</div>

In [ ]:
def z(x, mu, sigma): return (... - ...) / ...

In [ ]:
neighbor_check('z')

**Always use the training mean and SD**, for the training molecules, the test molecules, and any new molecule. The test set has to be measured on the same scale as the training set. `train_su` does exactly that: it is `z`, with $\mu$ and $\sigma$ taken from the **training** column.

In [ ]:
def train_su(x, feature): return z(x, np.mean(train_roh.column(feature)), np.std(train_roh.column(feature)))

Now build the standardized tables. Each line is one column: the bp stays in kelvin; each feature goes into standard units with `train_su`.

In [ ]:
train = Table().with_columns(
    'bp',      train_roh.column('bp'),
    'MW',      train_su(train_roh.column('MW'), 'MW'),
    'degree',  train_su(train_roh.column('degree'), 'degree'),
    'carbons', train_su(train_roh.column('carbons'), 'carbons'))
train.show(5)

In [ ]:
test = Table().with_columns(
    'bp',      test_roh.column('bp'),
    'MW',      train_su(test_roh.column('MW'), 'MW'),
    'degree',  train_su(test_roh.column('degree'), 'degree'),
    'carbons', train_su(test_roh.column('carbons'), 'carbons'))
test.show(5)

In standard units, each training feature has mean 0 and SD 1. (A mean like `-5.5e-16` means −5.5 × 10⁻¹⁶: zero, up to rounding.) The test features come out close to 0 and 1, but not exactly: they were scaled with the training numbers.

In [ ]:
np.mean(train.column('MW')), np.std(train.column('MW'))

---
### 🕵️ The mystery molecule
Now your team builds the whole method for **one mystery molecule**: the first molecule in the test set. Its boiling point stays hidden until Challenge 7.

The idea ([Inferential Thinking 17.4](https://inferentialthinking.com/chapters/17/4/Implementing_the_Classifier.html)): find the training molecules **closest** to the mystery molecule, and predict that it boils at about the **average** of their boiling points.

We'll use two features, MW and degree. `mystery` is the molecule's features in standard units, as an array.

In [ ]:
features = ['MW', 'degree']
mystery = np.array(test.select(features).row(0))
mystery

For comparison, the mystery molecule in its original units (bp hidden):

In [ ]:
test_roh.select(features).take(0)

### 📐 Distance
The distance between two molecules with features $(x_1, y_1)$ and $(x_2, y_2)$ is the straight-line distance:

$$d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

With arrays, `pt1 - pt2` subtracts **every feature at once**, so one line works for any number of features.

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 3 of 11 — The distance formula</strong><br>
The square root of the <strong>sum</strong> of the <strong>squared</strong> differences.
</div>

In [ ]:
def distance(pt1, pt2): return np.sqrt(np.sum((... - ...) ** 2))

In [ ]:
neighbor_check('distance')

`train.select(features).row(0)` is the first training molecule's features. `np.array(...)` turns that row into an array, so it can be subtracted from `mystery`.

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 4 of 11 — One distance</strong><br>
How far is the <strong>first training molecule</strong> from the mystery molecule? Give the second point.
</div>

In [ ]:
d0 = distance(np.array(train.select(features).row(0)), ...)

In [ ]:
neighbor_check('d0')

Now **every** training molecule, with a list comprehension. `train.select(features).rows` is all the training rows; the comprehension goes through them one at a time, calling each one `row`.

*"The distance from `row` to the mystery molecule, for each `row` of the training features."*

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 5 of 11 — Every distance</strong><br>
Fill in the second point, and which columns to select.
</div>

In [ ]:
all_dists = [distance(np.array(row), ...) for row in train.select(...).rows]

In [ ]:
neighbor_check('all_dists')

### 👥 The nearest neighbors
Three Table steps, chained in one line:
1. `train.with_column('Distance', all_dists)` adds the distances as a column,
2. `.sort('Distance')` puts the closest first,
3. `.take(np.arange(5))` keeps rows 0 to 4: the **5 nearest neighbors** (k = 5).

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 6 of 11 — The 5 nearest neighbors</strong><br>
Which list of distances, and which column to sort by?
</div>

In [ ]:
nearest = train.with_column('Distance', ...).sort(...).take(np.arange(5))

In [ ]:
neighbor_check('nearest')

In [ ]:
nearest

**See the neighbors.** Every training molecule in standard units (gray), the 5 nearest neighbors (red), and the mystery molecule (★). Degree is 1, 2 or 3, so the molecules sit in three rows. Count the red points: if fewer than 5 show, some neighbors are hiding under the ★, at distance 0.

In [ ]:
plt.scatter(train.column('MW'), train.column('degree'), color='gray', alpha=0.5, label='training molecules')
plt.scatter(nearest.column('MW'), nearest.column('degree'), color='#9E1B34', s=120, label='5 nearest')
plt.scatter(mystery.item(0), mystery.item(1), marker='*', color='gold', edgecolors='black', s=400, label='mystery molecule')
plt.xlabel('MW (standard units)'); plt.ylabel('degree (standard units)'); plt.legend();

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 7 of 11 — The prediction</strong><br>
Predict the mystery molecule's bp as the <strong>average</strong> bp of its 5 neighbors.
</div>

In [ ]:
prediction = np.average(nearest.column(...))

In [ ]:
neighbor_check('prediction')

---
### 🔧 Your toolkit
That's the whole method! Two short functions package **the same lines you just wrote**, so they work for any molecule and any `k`. The competition rounds use them.

**`closest(point, k)`**: Challenges 5 and 6. The `k` training molecules nearest to `point`, as a table.

In [ ]:
def closest(point, k):
    dists = [distance(np.array(row), point) for row in train.select(features).rows]
    return train.with_column('Distance', dists).sort('Distance').take(np.arange(k))

**`predict_knn(i, k)`**: Challenge 7, for test molecule number `i`. It takes that molecule's features, finds its `k` nearest neighbors, and averages their boiling points.

In [ ]:
def predict_knn(i, k):
    point = np.array(test.select(features).row(i))
    return np.average(closest(point, k).column('bp'))

In [ ]:
neighbor_check('toolkit')

---
## 🌿 Tribute Round — Run exactly as written

### 📏 How a round is scored
For every test molecule, the **error** is actual bp − predicted bp. To combine all the errors into one score:

1. **SSE**, the sum of squared errors: square each error, then add them up. Squaring makes every error positive, so errors above and below can't cancel out, and big misses count extra.
$$\text{SSE} = \sum_{i=1}^{n} (\text{actual}_i - \text{predicted}_i)^2$$
2. **MSE**, the mean squared error: SSE divided by $n$, the number of test molecules. It's the average squared error, so it doesn't grow just because there are more molecules.
$$\text{MSE} = \frac{\text{SSE}}{n}$$
3. **RMSE**, the root mean squared error: the square root of the MSE. Its units are back in **kelvin**, so it reads as the size of a typical miss.
$$\text{RMSE} = \sqrt{\text{MSE}}$$

**Lower is better.** This is the same idea as the error that least squares makes as small as possible in regression ([Inferential Thinking 15.3](https://inferentialthinking.com/chapters/15/3/Method_of_Least_Squares.html)). Each line of `rmse` is one step:

In [ ]:
def rmse(predictions):
    errors = test.column('bp') - predictions
    sse = np.sum(errors ** 2)
    mse = sse / len(errors)
    return np.sqrt(mse)

**The Tribute Round:** predict every test molecule with MW only and k = 5. The list comprehension runs `predict_knn` once for each row number `i` of the test set.

In [ ]:
features = ['MW']
tribute_predictions = [predict_knn(i, 5) for i in np.arange(test.num_rows)]
tribute_rmse = rmse(tribute_predictions)
tribute_rmse

**Predicted vs. actual.** A perfect prediction lands on the dashed line.

In [ ]:
results = Table().with_columns('actual bp', test.column('bp'), 'predicted bp', tribute_predictions)
results.scatter('actual bp', 'predicted bp')
plt.plot([300, 550], [300, 550], '--', color='gray');

#### 📡 Submit your Tribute score

In [ ]:
submit_button('Tribute', 'tribute_rmse')

---
## ⚔️ Arena Round — Tune Your Model!
**Choose your settings.** Available features: `'MW'`, `'degree'`, `'carbons'`. Change them and `k`, then rerun Challenge 8 and the cells after it. Your **best** Arena score counts.

In [ ]:
features = ['MW', 'degree']   # try adding 'carbons'
k = 5                         # try 1, 3, 7, 10, 15 …

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 8 of 11 — Predict every test molecule</strong><br>
The same list comprehension as the Tribute Round, with your own <code>k</code>. <code>i</code> is the row number of each test molecule; how many rows does the test set have?
</div>

In [ ]:
arena_predictions = [predict_knn(..., k) for i in np.arange(...)]

In [ ]:
neighbor_check('arena')

In [ ]:
arena_rmse = rmse(arena_predictions)
arena_rmse

In [ ]:
results = Table().with_columns('actual bp', test.column('bp'), 'predicted bp', arena_predictions)
results.scatter('actual bp', 'predicted bp')
plt.plot([300, 550], [300, 550], '--', color='gray');

**Scouting report: which k?** `rmse_for_k(kk)` scores the Arena predictions for one value of k. The list comprehension tries k = 1 to 20 with your current features, and the table shows the results. Small k follows single neighbors (noisy); large k averages over molecules that aren't really alike.

In [ ]:
def rmse_for_k(kk): return rmse([predict_knn(i, kk) for i in np.arange(test.num_rows)])

In [ ]:
sweep = Table().with_columns('k', np.arange(1, 21), 'RMSE', [rmse_for_k(kk) for kk in np.arange(1, 21)])
sweep.plot('k', 'RMSE')
sweep.sort('RMSE').show(3)

#### 📡 Submit your Arena score

In [ ]:
submit_button('Arena', 'arena_rmse', extra_fn=lambda: {'features': features, 'k': int(k)})

---
## 🎯 Prize Round — Predict Real Molecules!

| Molecule | MW (g/mol) | Degree | Carbons | Actual bp |
|---|---|---|---|---|
| **Ethanol** | 46.07 | 1 | 2 | 351.4 K |
| **Isopropanol (IPA)** | 60.10 | 2 | 3 | 355.4 K |

A new molecule goes into standard units with the **training** mean and SD: `train_su`. (`standard_units()` won't do: it would use the mean and SD of just this one molecule.)

Set your best features and k from the Arena:

In [ ]:
features = ['MW', 'degree']   # ← your best features
k = 5                         # ← your best k

Each molecule's raw features are stored by name, so `ethanol_raw['MW']` is 46.07.

**Ethanol, the worked example:** a list comprehension puts each of your features in standard units, *"`train_su` of the ethanol value, for each feature `f`."*

In [ ]:
ethanol_raw = {'MW': 46.07, 'degree': 1, 'carbons': 2}
ipa_raw     = {'MW': 60.10, 'degree': 2, 'carbons': 3}

In [ ]:
ethanol_std = np.array([train_su(ethanol_raw[f], f) for f in features])
ethanol_std

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 9 of 11 — Standardize isopropanol</strong><br>
The same pattern as ethanol. What goes second in <code>train_su</code>?
</div>

In [ ]:
ipa_std = np.array([train_su(ipa_raw[f], ...) for f in features])

In [ ]:
neighbor_check('ipa_std')

Predict each molecule as the average bp of its `k` nearest neighbors, and find the error.

In [ ]:
ethanol_pred = np.average(closest(ethanol_std, k).column('bp'))
ethanol_error = abs(ethanol_pred - 351.4)
ethanol_pred, ethanol_error

In [ ]:
ipa_pred = np.average(closest(ipa_std, k).column('bp'))
ipa_error = abs(ipa_pred - 355.4)
ipa_pred, ipa_error

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 10 of 11 — The Prize score</strong><br>
The Prize score is the <strong>average</strong> of the two errors.
</div>

In [ ]:
prize_score = (ethanol_error + ipa_error) / ...

In [ ]:
neighbor_check('prize_score')

In [ ]:
prize = Table().with_columns('molecule', make_array('Ethanol', 'Isopropanol'),
                             'actual bp', make_array(351.4, 355.4),
                             'predicted bp', make_array(ethanol_pred, ipa_pred))
prize.barh('molecule')

#### 📡 Submit your Prize score

In [ ]:
submit_button('Prize', 'prize_score')

---
## 🌟 Wildcard Round — Inverse-Distance Weighting
In plain kNN, all 5 neighbors get an equal vote, even one much farther away than the others. **Weighting** gives each neighbor a vote of 1 / distance, so the closest count the most:

$$\text{weighted prediction} = \frac{\sum w_i \cdot \text{bp}_i}{\sum w_i}, \qquad w_i = \frac{1}{d_i}$$

Back to the mystery molecule's `nearest` table from Challenge 6:

In [ ]:
nearest

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Challenge 11 of 11 — The weights</strong><br>
One weight per neighbor: 1 divided by its distance. The tiny <code>1e-9</code> keeps us from dividing by zero when a neighbor is at distance 0.
</div>

In [ ]:
weights = 1 / (nearest.column(...) + 1e-9)

In [ ]:
neighbor_check('weights')

In [ ]:
weighted_prediction = np.sum(weights * nearest.column('bp')) / np.sum(weights)
prediction, weighted_prediction

A neighbor at distance 0 has the same MW and degree as the mystery molecule: an isomer. Its weight is huge, so it essentially decides the weighted prediction by itself. Is that a good thing here?

**`predict_weighted(i, k)`** is `predict_knn` with your weights: the same neighbors, a weighted average.

In [ ]:
def predict_weighted(i, k):
    near = closest(np.array(test.select(features).row(i)), k)
    w = 1 / (near.column('Distance') + 1e-9)
    return np.sum(w * near.column('bp')) / np.sum(w)

Now the whole test set, weighted. Run as written.

In [ ]:
features = ['MW', 'degree']
wildcard_predictions = [predict_weighted(i, 5) for i in np.arange(test.num_rows)]
wildcard_rmse = rmse(wildcard_predictions)
wildcard_rmse

#### 📡 Submit your Wildcard score

In [ ]:
submit_button('Wildcard', 'wildcard_rmse')

---
## 💬 Reflection

1. **In Challenge 3, why do we square the differences before adding them?** What would go wrong if we just added `pt1 - pt2`?
2. **Why do we use the training mean and SD for the test set and for new molecules?**
3. **RMSE squares the errors, takes the mean, then the square root. Why not just average the errors?**
4. **In the Arena, did adding more features improve your RMSE? Did a bigger k always help?**
5. **When would weighting by 1 / distance help most?**

*Your answers here:*

1.

2.

3.

4.

5. 